# 02 - Behavioral Validation

Do the fear / greed / herd indicators actually relate to future returns, or are they noise? We score each against the next-day return.

## How each indicator is defined

All three are derived from the same OHLCV data (no external feed) and normalized with an **expanding z-score**, which uses only past-and-present values so no future information leaks backwards.

- **Fear - downside deviation.** The proposal frames fear as *high volatility + negative returns*. We capture both in one quantity: the root-mean-square of negative daily returns over the window. This is ordinary volatility restricted to losing days (the semi-deviation behind the Sortino ratio), so a calm market and a steadily rising market both read as low fear, while frequent, large drops read as high fear. We use this rather than literally multiplying volatility by negative returns because it is a recognised risk measure and avoids double-counting the two-sided `volatility_20` feature the model already has.
- **Greed - momentum x relative volume.** Positive momentum scaled by how busy trading is versus its recent average; rising prices that the crowd is piling into.
- **Herd - joint extremes.** Fires only when an unusual volume spike and an unusually large price move (both above a z-score threshold) happen on the same day.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))  # so 'import src...' works from notebooks/

import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config
config = load_config('../config.yaml')
ticker = config['data']['tickers'][0]
config['data']['raw_dir'] = '../data/raw'  # paths are relative to project root
ticker

In [ ]:
from src.data.loader import load_ohlcv
from src.data.cleaner import clean_ohlcv
from src.behavioral import compute_fear, compute_greed, compute_herd
from src.behavioral.validation import validate_indicator

raw = load_ohlcv(ticker, config['data']['start_date'], config['data']['end_date'],
                 raw_dir=config['data']['raw_dir'])
df = clean_ohlcv(raw)

## Build indicators and the forward return they're scored against

In [ ]:
bcfg = config['behavioral']
indicators = {
    'fear': compute_fear(df, **bcfg['fear']),
    'greed': compute_greed(df, **bcfg['greed']),
    'herd': compute_herd(df, **bcfg['herd']),
}
forward_return = df['Close'].shift(-1) / df['Close'] - 1.0

## Diagnostics

`ic` is the rank correlation with next-day return; `top`/`bottom` are the mean forward return in the highest vs lowest readings.

In [ ]:
import pandas as pd
report = {name: validate_indicator(series, forward_return) for name, series in indicators.items()}
pd.DataFrame(report).T